## 데이터 로드, 형식 정리

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path('data/raw/SeoulBikeData.csv')
CLEAN_PATH = Path('data/processed/seoul_bike_clean.csv')

# 원본은 UTF-8이 아님
df = pd.read_csv(RAW_PATH, encoding='latin-1')
print(df.shape)
df.head()

(8760, 14)


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [2]:
# 컬럼명을 영문 소문자로 변경 (°, 공백, 괄호, 단위 제거)
COLUMN_NAMES = {
    'Date': 'date',
    'Rented Bike Count': 'rented_bike_count',
    'Hour': 'hour',
    'Temperature(°C)': 'temperature',
    'Humidity(%)': 'humidity',
    'Wind speed (m/s)': 'wind_speed',
    'Visibility (10m)': 'visibility',
    'Dew point temperature(°C)': 'dew_point_temp',
    'Solar Radiation (MJ/m2)': 'solar_radiation',
    'Rainfall(mm)': 'rainfall',
    'Snowfall (cm)': 'snowfall',
    'Seasons': 'seasons',
    'Holiday': 'holiday',
    'Functioning Day': 'functioning_day',
}
df = df.rename(columns=COLUMN_NAMES)
df.columns.tolist()

['date',
 'rented_bike_count',
 'hour',
 'temperature',
 'humidity',
 'wind_speed',
 'visibility',
 'dew_point_temp',
 'solar_radiation',
 'rainfall',
 'snowfall',
 'seasons',
 'holiday',
 'functioning_day']

In [3]:
# Date는 dd/mm/yyyy 문자열 -> datetime
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
print(df['date'].min(), '~', df['date'].max())
df.dtypes

2017-12-01 00:00:00 ~ 2018-11-30 00:00:00


date                 datetime64[us]
rented_bike_count             int64
hour                          int64
temperature                 float64
humidity                      int64
wind_speed                  float64
visibility                    int64
dew_point_temp              float64
solar_radiation             float64
rainfall                    float64
snowfall                    float64
seasons                         str
holiday                         str
functioning_day                 str
dtype: object

In [4]:
# 정리본을 UTF-8로 저장 후 다시 읽어서 확인
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8')

check = pd.read_csv(CLEAN_PATH, parse_dates=['date'])
print(check.shape)
print(check.equals(df))

(8760, 14)
True


## 3. EDA

In [ ]:
import matplotlib.pyplot as plt

# 운영하지 않은 시간(functioning_day == 'No')은 대여 수가 전부 0이라 평균을 왜곡하므로
# 수요 관련 그래프는 운영한 시간만으로 그림 (df 자체는 변경하지 않음)
op = df[df['functioning_day'] == 'Yes']
print(df['functioning_day'].value_counts())
print('운영 시간 중 대여 수 0인 행:', (op['rented_bike_count'] == 0).sum())
op.describe()

In [ ]:
# 타깃 분포
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(op['rented_bike_count'], bins=50)
ax.set_xlabel('Rented bike count (per hour)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of hourly rentals')
plt.show()
print('skewness:', round(op['rented_bike_count'].skew(), 3))

In [ ]:
# 시간대별 평균 대여 수
hourly = op.groupby('hour')['rented_bike_count'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly.index, hourly.values, marker='o')
ax.set_xticks(range(24))
ax.set_xlabel('Hour')
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by hour')
ax.grid(alpha=0.3)
plt.show()
hourly.round(1)

In [ ]:
# 계절별 평균 대여 수
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
seasonal = op.groupby('seasons')['rented_bike_count'].mean().reindex(season_order)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(seasonal.index, seasonal.values)
ax.set_ylabel('Mean rented bike count')
ax.set_title('Mean rentals by season')
plt.show()
seasonal.round(1)

In [ ]:
# 기온과 대여 수
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(op['temperature'], op['rented_bike_count'], s=3, alpha=0.3)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Rented bike count')
ax.set_title('Temperature vs rentals')
plt.show()

# 5°C 구간별 평균
temp_bin = pd.cut(op['temperature'], bins=range(-20, 45, 5))
op.groupby(temp_bin, observed=True)['rented_bike_count'].agg(['mean', 'count']).round(1)

In [ ]:
# 비·눈 여부에 따른 평균 대여 수
weather = pd.DataFrame({
    'rain': op.groupby(op['rainfall'] > 0)['rented_bike_count'].mean(),
    'snow': op.groupby(op['snowfall'] > 0)['rented_bike_count'].mean(),
}).rename(index={False: 'No', True: 'Yes'})

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, col in zip(axes, ['rain', 'snow']):
    ax.bar(weather.index, weather[col])
    ax.set_title(f'{col} > 0')
axes[0].set_ylabel('Mean rented bike count')
plt.show()
print('rainfall > 0 시간:', (op['rainfall'] > 0).sum(), '/ snowfall > 0 시간:', (op['snowfall'] > 0).sum())
weather.round(1)

In [ ]:
# 공휴일/평일, 주말/주중의 시간대별 패턴 (요일은 그래프용으로만 계산, df에는 추가하지 않음)
day_type = (op['date'].dt.dayofweek >= 5).map({False: 'Weekday', True: 'Weekend'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for label, g in op.groupby('holiday'):
    axes[0].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
for label, g in op.groupby(day_type):
    axes[1].plot(g.groupby('hour')['rented_bike_count'].mean(), marker='o', label=label)
axes[0].set_title('Holiday vs No Holiday')
axes[1].set_title('Weekday vs Weekend')
for ax in axes:
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel('Hour')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Mean rented bike count')
plt.show()

print(op.groupby('holiday')['rented_bike_count'].mean().round(1))
print(op.groupby(day_type)['rented_bike_count'].mean().round(1))

In [ ]:
# 수치형 변수 간 상관관계
corr = op.select_dtypes('number').corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()
corr['rented_bike_count'].drop('rented_bike_count').sort_values(key=abs, ascending=False).round(3)

In [ ]:
# humidity 최솟값 0 확인 (이상치 여부)
print('humidity == 0 행 수:', (df['humidity'] == 0).sum())
print('humidity 하위 값:', sorted(df['humidity'].unique())[:10])
df[df['humidity'] == 0][['date', 'hour', 'temperature', 'humidity', 'dew_point_temp', 'rainfall', 'rented_bike_count']]

## 4. 전처리

In [ ]:
import numpy as np

# humidity == 0 인 17행은 같은 행의 기온·이슬점과 맞지 않음 (측정 오류로 추정)
# → 기온·이슬점으로 상대습도를 계산(Magnus 공식)해서 대체
def humidity_from_dew_point(temp, dew_point):
    a, b = 17.625, 243.04
    return 100 * np.exp(a * dew_point / (b + dew_point)) / np.exp(a * temp / (b + temp))

zero_humidity = df['humidity'] == 0
estimated = humidity_from_dew_point(df.loc[zero_humidity, 'temperature'], df.loc[zero_humidity, 'dew_point_temp'])
df.loc[zero_humidity, 'humidity'] = estimated.round().astype(int)

print('보정한 행 수:', zero_humidity.sum())
print('보정 후 humidity == 0 행 수:', (df['humidity'] == 0).sum())
df.loc[zero_humidity, ['date', 'hour', 'temperature', 'dew_point_temp', 'humidity']]